# Privacy-preserving federated learning: serial C++ vs CUDA
**Dataset:** PneumoniaMNIST (28x28 chest X-rays, pneumonia vs normal), **model:** logistic regression, **5 clients** (each keeps its own data; only model updates reach the server).

Two privacy tools are implemented, each once in serial C++ and once in CUDA:
- **Differential privacy (DP) gradient:** clip each example's gradient, then add Gaussian noise.
- **Secure aggregation:** each client adds random masks to its update; the masks cancel when the server sums everything, so the server only ever sees the sum.

Experiments:
1. **Correctness + speed**: do serial and CUDA agree, and how much faster is CUDA?
2. **Federated training** with DP + secure aggregation (no privacy, sigma = 1, 3, 6).
3. **Model poisoning**: one malicious client against mean (secure aggregation) vs median.

## Step 1 - Prepare the data
Load the `.npz`, flatten images to 784 numbers, standardise each image on its own (purely local, so it leaks nothing across clients), and save as raw binary files the C++ program can read.

In [5]:
import numpy as np, os, glob
path = glob.glob('/kaggle/input/**/pneumoniamnist*.npz', recursive=True)[0]
d = np.load(path)

os.makedirs('data', exist_ok=True)
for split in ['train', 'test']:
    X = d[f'{split}_images'].reshape(-1, 784).astype(np.float32) / 255.0
    X = (X - X.mean(1, keepdims=True)) / (X.std(1, keepdims=True) + 1e-6) * 0.25
    y = d[f'{split}_labels'].reshape(-1).astype(np.uint8)
    print(split, X.shape, np.bincount(y))
    X.tofile(f'data/{split}_x.bin')
    y.tofile(f'data/{split}_y.bin')

train (3882, 784) [ 388 3494]
test (624, 784) [234 390]


## Step 2 - The C++ / CUDA program
The file is organised in the order you would read it:

| Part | What it does |
|---|---|
| **Serial code** | `dp_grad_serial` (clipped gradient) and `secagg_serial` (masked sum), plain C++ |
| **CUDA kernels** | `residual_kernel` + `grad_kernel` (DP gradient) and `secagg_kernel` (secure aggregation) |
| **`train()`** | One federated training loop; a `gpu` flag chooses serial or CUDA, so both share the same logic |
| **Experiments** | correctness check, speed benchmark, then the federated runs |

Key idea of secure aggregation: client *i* adds a mask for each higher-numbered partner and subtracts it for each lower-numbered one, so all masks cancel in the sum.

**2a. Setup:** includes, constants (clients, rounds, local steps), data structures and the data loader. This cell *creates* `fedml.cu`; the next cells append to it, so run them in order.

In [6]:
%%writefile fedml.cu
// Privacy-preserving federated learning: serial C++ vs CUDA (compact version)
#include <cstdio>
#include <cstdlib>
#include <cstdint>
#include <cstring>
#include <cmath>
#include <string>
#include <vector>
#include <numeric>
#include <algorithm>
#include <random>
#include <chrono>

#define CK(x) do { cudaError_t e_ = (x); if (e_ != cudaSuccess) { fprintf(stderr, "CUDA error: %s (line %d)\n", cudaGetErrorString(e_), __LINE__); exit(1); } } while (0)

constexpr int D = 784, P = D + 1, TPB = 256;          // 784 pixels + 1 bias = 785 parameters
constexpr int K = 5, ROUNDS = 30, LOCAL = 4;          // clients, federated rounds, local steps per round
constexpr float SCALE = 65536.f;                      // fixed-point scale for secure aggregation
constexpr float BOOST = 5.f;                          // malicious client sends -BOOST x its honest update

struct Data { std::vector<float> X; std::vector<uint8_t> y; int n = 0; };
struct Setting { const char* name; float sigma, clip, lr; bool poison, median; };  // sigma: noise multiplier, clip 0 = no clipping

template <class T> T* dalloc(size_t n) { T* p; CK(cudaMalloc(&p, n * sizeof(T))); return p; }
template <class F> static double ms(F f) {
    auto t = std::chrono::steady_clock::now(); f();
    return std::chrono::duration<double, std::milli>(std::chrono::steady_clock::now() - t).count();
}

static bool load(const std::string& dir, const std::string& split, Data& d) {
    FILE* fx = fopen((dir + "/" + split + "_x.bin").c_str(), "rb");
    FILE* fy = fopen((dir + "/" + split + "_y.bin").c_str(), "rb");
    if (!fx || !fy) return false;
    fseek(fy, 0, SEEK_END); d.n = (int)ftell(fy); fseek(fy, 0, SEEK_SET);
    d.y.resize(d.n); d.X.resize((size_t)d.n * D);
    bool ok = fread(d.y.data(), 1, d.n, fy) == (size_t)d.n && fread(d.X.data(), sizeof(float), d.X.size(), fx) == d.X.size();
    fclose(fx); fclose(fy);
    return ok;
}


Overwriting fedml.cu


**2b. Serial C++:** secure-aggregation helpers (masks that cancel in the sum), the DP gradient with per-example clipping, accuracy, median aggregation and the privacy budget (epsilon).

In [7]:
%%writefile -a fedml.cu

// ======================= 1. SERIAL C++ =======================

// Secure aggregation helpers, shared by serial code and the CUDA kernel (demo hash, NOT a real cryptographic PRG)
__host__ __device__ inline uint32_t mix(uint32_t x) {
    x ^= x >> 16; x *= 0x7feb352dU; x ^= x >> 15; x *= 0x846ca68bU; x ^= x >> 16; return x;
}
__host__ __device__ inline uint32_t pair_mask(uint32_t seed, int a, int b, int d) {      // mask shared by clients a < b
    return mix(mix(seed ^ (uint32_t)(a * 4096 + b)) + (uint32_t)d);
}
// what client i sends for coordinate d: fixed-point update + masks (+ for higher partners, - for lower ones)
__host__ __device__ inline uint32_t client_share(const float* U, int Dm, uint32_t seed, int i, int d) {
    uint32_t v = (uint32_t)(int32_t)lrintf(U[(size_t)i * Dm + d] * SCALE);
    for (int j = 0; j < K; ++j) {
        if (j > i) v += pair_mask(seed, i, j, d);
        if (j < i) v -= pair_mask(seed, j, i, d);
    }
    return v;
}
// server adds all shares: the masks cancel and what is left is the mean of the true updates
__host__ __device__ inline float secagg_coord(const float* U, int Dm, uint32_t seed, int d) {
    uint32_t s = 0;
    for (int i = 0; i < K; ++i) s += client_share(U, Dm, seed, i, d);
    return (float)(int32_t)s / SCALE / (float)K;
}
static void secagg_serial(const float* U, int Dm, uint32_t seed, float* out) {
    for (int d = 0; d < Dm; ++d) out[d] = secagg_coord(U, Dm, seed, d);
}

// sum of clipped per-example logistic-regression gradients (clip <= 0: no clipping)
static void dp_grad_serial(const float* X, const uint8_t* y, const float* w, int N, float clip, float* g) {
    std::fill(g, g + P, 0.f);
    for (int i = 0; i < N; ++i) {
        const float* x = X + (size_t)i * D;
        float dot = w[D], n2 = 1.f;                                    // bias = a feature equal to 1
        for (int j = 0; j < D; ++j) { dot += w[j] * x[j]; n2 += x[j] * x[j]; }
        float r = 1.f / (1.f + expf(-dot)) - (float)y[i];              // prediction - label
        float c = clip > 0.f ? fminf(1.f, clip / (fabsf(r) * sqrtf(n2) + 1e-12f)) : 1.f;
        for (int j = 0; j < D; ++j) g[j] += c * r * x[j];
        g[D] += c * r;
    }
}

static float accuracy(const Data& d, const float* w) {
    int ok = 0;
    for (int i = 0; i < d.n; ++i) {
        float z = w[D];
        for (int j = 0; j < D; ++j) z += w[j] * d.X[(size_t)i * D + j];
        ok += ((z > 0.f) == (d.y[i] == 1));
    }
    return (float)ok / (float)d.n;
}

// median needs to SEE each client's update, so it cannot be combined with secure aggregation
static void median_agg(const float* U, int Dm, float* out) {
    std::vector<float> v(K);
    for (int d = 0; d < Dm; ++d) {
        for (int k = 0; k < K; ++k) v[k] = U[(size_t)k * Dm + d];
        std::sort(v.begin(), v.end());
        out[d] = v[K / 2];
    }
}

// epsilon of the Gaussian mechanism (zCDP accounting)
static double dp_epsilon(double sigma, int steps, double delta) {
    double rho = steps / (2.0 * sigma * sigma);
    return rho + 2.0 * std::sqrt(rho * std::log(1.0 / delta));
}


Appending to fedml.cu


**2c. CUDA kernels:** `residual_kernel` and `grad_kernel` compute the DP gradient on the GPU, `secagg_kernel` does secure aggregation (one thread per coordinate), and `GpuGrad` manages the GPU buffers.

In [8]:
%%writefile -a fedml.cu

// ======================= 2. CUDA KERNELS =======================

// one block per example: prediction - label, and the example's norm
__global__ void residual_kernel(const float* X, const uint8_t* y, const float* w, float* r, float* xnorm) {
    int b = blockIdx.x;
    const float* x = X + (size_t)b * D;
    float dot = 0.f, n2 = 0.f;
    for (int j = threadIdx.x; j < D; j += blockDim.x) { dot += w[j] * x[j]; n2 += x[j] * x[j]; }
    __shared__ float sd[TPB], sn[TPB];
    sd[threadIdx.x] = dot; sn[threadIdx.x] = n2;
    __syncthreads();
    for (int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (threadIdx.x < s) { sd[threadIdx.x] += sd[threadIdx.x + s]; sn[threadIdx.x] += sn[threadIdx.x + s]; }
        __syncthreads();
    }
    if (threadIdx.x == 0) {
        r[b] = 1.f / (1.f + expf(-(sd[0] + w[D]))) - (float)y[b];
        xnorm[b] = sqrtf(sn[0] + 1.f);
    }
}

// one thread per parameter: sum the clipped per-example gradients
__global__ void grad_kernel(const float* X, const float* r, const float* xnorm, int N, float clip, float* g) {
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j >= P) return;
    float acc = 0.f;
    for (int b = 0; b < N; ++b) {
        float c = clip > 0.f ? fminf(1.f, clip / (fabsf(r[b]) * xnorm[b] + 1e-12f)) : 1.f;
        acc += c * r[b] * (j < D ? X[(size_t)b * D + j] : 1.f);
    }
    g[j] = acc;
}

// one thread per coordinate; same secagg_coord() as the serial code
__global__ void secagg_kernel(const float* U, int Dm, uint32_t seed, float* out) {
    int d = blockIdx.x * blockDim.x + threadIdx.x;
    if (d < Dm) out[d] = secagg_coord(U, Dm, seed, d);
}

// GPU buffers for the DP gradient of up to n examples (data is uploaded once)
struct GpuGrad {
    float *X, *w, *g, *r, *xn; uint8_t* y;
    void init(const float* hx, const uint8_t* hy, int n) {
        X = dalloc<float>((size_t)n * D); y = dalloc<uint8_t>(n); w = dalloc<float>(P);
        g = dalloc<float>(P); r = dalloc<float>(n); xn = dalloc<float>(n);
        CK(cudaMemcpy(X, hx, (size_t)n * D * sizeof(float), cudaMemcpyHostToDevice));
        CK(cudaMemcpy(y, hy, n, cudaMemcpyHostToDevice));
    }
    void run(const float* hw, int n, float clip, float* hg) {          // upload w, run kernels, download g
        CK(cudaMemcpy(w, hw, P * sizeof(float), cudaMemcpyHostToDevice));
        residual_kernel<<<n, TPB>>>(X, y, w, r, xn);
        grad_kernel<<<(P + TPB - 1) / TPB, TPB>>>(X, r, xn, n, clip, g);
        CK(cudaMemcpy(hg, g, P * sizeof(float), cudaMemcpyDeviceToHost));
    }
    void release() { for (void* p : {(void*)X, (void*)y, (void*)w, (void*)g, (void*)r, (void*)xn}) cudaFree(p); }
};


Appending to fedml.cu


**2d. Federated training:** one `train()` function for both backends. The `gpu` flag only switches which gradient and aggregation code runs.

In [9]:
%%writefile -a fedml.cu

// ======================= 3. FEDERATED TRAINING (serial or CUDA) =======================

static std::vector<float> train(const Setting& s, const std::vector<Data>& cl, const Data& test, bool gpu) {
    std::mt19937 rng(1);
    std::normal_distribution<float> gauss(0.f, 1.f);
    std::vector<GpuGrad> gg(K);
    float *dU = nullptr, *dout = nullptr;
    if (gpu) {
        for (int k = 0; k < K; ++k) gg[k].init(cl[k].X.data(), cl[k].y.data(), cl[k].n);
        dU = dalloc<float>((size_t)K * P); dout = dalloc<float>(P);
    }
    std::vector<float> w(P, 0.f), wl(P), g(P), U((size_t)K * P), agg(P), acc;
    for (int round = 0; round < ROUNDS; ++round) {
        for (int k = 0; k < K; ++k) {                                  // each client trains on ITS OWN data
            wl = w;
            for (int step = 0; step < LOCAL; ++step) {
                if (gpu) gg[k].run(wl.data(), cl[k].n, s.clip, g.data());
                else     dp_grad_serial(cl[k].X.data(), cl[k].y.data(), wl.data(), cl[k].n, s.clip, g.data());
                for (int j = 0; j < P; ++j) {
                    float noise = s.sigma > 0.f ? s.sigma * s.clip * gauss(rng) : 0.f;   // DP noise
                    wl[j] -= s.lr * (g[j] + noise) / (float)cl[k].n;
                }
            }
            float m = (s.poison && k == 0) ? -BOOST : 1.f;            // client 0 flips and boosts its update
            for (int j = 0; j < P; ++j) U[(size_t)k * P + j] = m * (wl[j] - w[j]);
        }
        if (s.median) median_agg(U.data(), P, agg.data());
        else if (gpu) {                                                // secure aggregation on the GPU
            CK(cudaMemcpy(dU, U.data(), U.size() * sizeof(float), cudaMemcpyHostToDevice));
            secagg_kernel<<<(P + TPB - 1) / TPB, TPB>>>(dU, P, 1000 + round, dout);
            CK(cudaMemcpy(agg.data(), dout, P * sizeof(float), cudaMemcpyDeviceToHost));
        } else secagg_serial(U.data(), P, 1000 + round, agg.data());
        for (int j = 0; j < P; ++j) w[j] += agg[j];
        acc.push_back(accuracy(test, w.data()));
    }
    if (gpu) { for (auto& x : gg) x.release(); cudaFree(dU); cudaFree(dout); }
    return acc;
}


Appending to fedml.cu


**2e. Experiments and `main`:** correctness check, speed benchmark, then federated runs with DP and with poisoning.

In [10]:
%%writefile -a fedml.cu

// ======================= 4. EXPERIMENTS =======================

static std::vector<float> randn(size_t n, float sd, unsigned seed) {
    std::mt19937 r(seed); std::normal_distribution<float> nd(0.f, sd);
    std::vector<float> v(n); for (auto& x : v) x = nd(r); return v;
}

// 1a. do serial and CUDA agree?
static void check_correctness(const Data& tr) {
    printf("\n=== 1a. Correctness ===\n");
    std::vector<float> w = randn(P, 0.05f, 0), gs(P), gc(P);
    GpuGrad gg; gg.init(tr.X.data(), tr.y.data(), tr.n);
    for (float clip : {1.0f, 0.0f}) {
        dp_grad_serial(tr.X.data(), tr.y.data(), w.data(), tr.n, clip, gs.data());
        gg.run(w.data(), tr.n, clip, gc.data());
        float err = 0.f, mx = 0.f;
        for (int j = 0; j < P; ++j) { err = std::max(err, fabsf(gs[j] - gc[j])); mx = std::max(mx, fabsf(gs[j])); }
        printf("DP gradient (clip=%.0f): max relative error %.1e  %s\n", clip, err / mx, err / mx < 1e-3f ? "OK" : "CHECK!");
    }
    gg.release();

    const int Dc = 200000;                                             // secure aggregation on random updates
    std::vector<float> U = randn((size_t)K * Dc, 0.1f, 5), s(Dc), c(Dc);
    secagg_serial(U.data(), Dc, 7, s.data());
    float *dU = dalloc<float>(U.size()), *dout = dalloc<float>(Dc);
    CK(cudaMemcpy(dU, U.data(), U.size() * sizeof(float), cudaMemcpyHostToDevice));
    secagg_kernel<<<(Dc + TPB - 1) / TPB, TPB>>>(dU, Dc, 7, dout);
    CK(cudaMemcpy(c.data(), dout, Dc * sizeof(float), cudaMemcpyDeviceToHost));
    cudaFree(dU); cudaFree(dout);
    float err = 0.f;
    for (int d = 0; d < Dc; ++d) {
        double m = 0; for (int i = 0; i < K; ++i) m += U[(size_t)i * Dc + d];
        err = std::max(err, (float)fabs(m / K - s[d]));
    }
    printf("Secure aggregation: max error vs plain mean %.1e (fixed-point bound %.1e)\n", err, 0.5f / SCALE);
    printf("Secure aggregation: serial and CUDA bit-identical: %s\n", memcmp(s.data(), c.data(), Dc * sizeof(float)) == 0 ? "yes" : "NO");
}

// 1b. speed: DP gradient and secure aggregation, serial vs CUDA (CUDA time includes host<->device copies)
static void bench(const Data& tr) {
    printf("\n=== 1b. Speed (best of 3, ms) ===\n%-28s %9s %9s %9s\n", "task", "serial", "CUDA", "speedup");
    const int Nmax = 100000;
    std::vector<float> X((size_t)Nmax * D), w = randn(P, 0.05f, 3), g(P);   // tile the real data
    std::vector<uint8_t> y(Nmax);
    for (int i = 0; i < Nmax; ++i) {
        memcpy(&X[(size_t)i * D], &tr.X[(size_t)(i % tr.n) * D], D * sizeof(float));
        y[i] = tr.y[i % tr.n];
    }
    GpuGrad gg; gg.init(X.data(), y.data(), Nmax);
    gg.run(w.data(), 1000, 1.f, g.data());                             // warm-up
    for (int N : {1000, 10000, 100000}) {
        double ts = 1e9, tc = 1e9;
        for (int r = 0; r < 3; ++r) {
            ts = std::min(ts, ms([&] { dp_grad_serial(X.data(), y.data(), w.data(), N, 1.f, g.data()); }));
            tc = std::min(tc, ms([&] { gg.run(w.data(), N, 1.f, g.data()); }));
        }
        char name[40]; snprintf(name, 40, "DP gradient, N=%d", N);
        printf("%-28s %9.2f %9.2f %8.1fx\n", name, ts, tc, ts / tc);
    }
    gg.release();
    for (int Dm : {100000, 1000000, 5000000}) {
        std::vector<float> U = randn((size_t)K * Dm, 0.1f, 9), out(Dm);
        float *dU = dalloc<float>(U.size()), *dout = dalloc<float>(Dm);
        int grid = (Dm + TPB - 1) / TPB;
        secagg_kernel<<<grid, TPB>>>(dU, Dm, 7, dout); CK(cudaDeviceSynchronize());
        double ts = 1e9, tc = 1e9;
        for (int r = 0; r < 3; ++r) {
            ts = std::min(ts, ms([&] { secagg_serial(U.data(), Dm, 7, out.data()); }));
            tc = std::min(tc, ms([&] {
                CK(cudaMemcpy(dU, U.data(), U.size() * sizeof(float), cudaMemcpyHostToDevice));
                secagg_kernel<<<grid, TPB>>>(dU, Dm, 7, dout);
                CK(cudaMemcpy(out.data(), dout, Dm * sizeof(float), cudaMemcpyDeviceToHost));
            }));
        }
        char name[40]; snprintf(name, 40, "Secure agg, %d params", Dm);
        printf("%-28s %9.2f %9.2f %8.1fx\n", name, ts, tc, ts / tc);
        cudaFree(dU); cudaFree(dout);
    }
}

// split the training set into K equal client shards
static std::vector<Data> split_clients(const Data& tr) {
    std::vector<int> perm(tr.n);
    std::iota(perm.begin(), perm.end(), 0);
    std::shuffle(perm.begin(), perm.end(), std::mt19937(42));
    int per = tr.n / K;
    std::vector<Data> cl(K);
    for (int k = 0; k < K; ++k) {
        cl[k].n = per;
        for (int i = 0; i < per; ++i) {
            int row = perm[k * per + i];
            cl[k].X.insert(cl[k].X.end(), tr.X.begin() + (size_t)row * D, tr.X.begin() + (size_t)(row + 1) * D);
            cl[k].y.push_back(tr.y[row]);
        }
    }
    return cl;
}

static void run_fl(const char* title, const std::vector<Setting>& list, const std::vector<Data>& cl, const Data& test) {
    printf("\n=== %s ===\n%-24s %8s %11s %9s %11s %9s %8s\n", title, "setting", "epsilon", "acc serial", "acc CUDA", "time serial", "time CUDA", "speedup");
    for (const Setting& s : list) {
        std::vector<float> a_s, a_c;
        double t_s = ms([&] { a_s = train(s, cl, test, false); }) / 1e3;
        double t_c = ms([&] { a_c = train(s, cl, test, true); }) / 1e3;
        double eps = s.sigma > 0.f ? dp_epsilon(s.sigma, ROUNDS * LOCAL, 1e-5) : INFINITY;
        printf("%-24s %8.1f %11.4f %9.4f %10.2fs %8.2fs %7.1fx\n", s.name, eps, a_s.back(), a_c.back(), t_s, t_c, t_s / t_c);
    }
}

int main(int argc, char** argv) {
    cudaDeviceProp prop;
    CK(cudaGetDeviceProperties(&prop, 0));
    CK(cudaFree(0));                                                   // create CUDA context before timing
    printf("GPU: %s\n", prop.name);
    Data train_set, test;
    if (argc < 2 || !load(argv[1], "train", train_set) || !load(argv[1], "test", test)) { fprintf(stderr, "Usage: %s <data_dir>\n", argv[0]); return 1; }
    printf("Data: %d train, %d test\n", train_set.n, test.n);

    check_correctness(train_set);
    bench(train_set);

    std::vector<Data> cl = split_clients(train_set);
    //                        name           sigma clip lr    poison median
    run_fl("2. Federated training with DP + secure aggregation", {
        {"no privacy",         0.f, 0.f, 0.1f, false, false},
        {"DP sigma=1",         1.f, 1.f, 1.0f, false, false},
        {"DP sigma=3",         3.f, 1.f, 1.0f, false, false},
        {"DP sigma=6",         6.f, 1.f, 1.0f, false, false}}, cl, test);
    run_fl("3. Model poisoning: client 0 sends -5x its update", {
        {"poison + mean (secagg)", 0.f, 0.f, 0.1f, true, false},
        {"poison + median",        0.f, 0.f, 0.1f, true, true}}, cl, test);
    printf("\nNote: median needs each client's update in the clear, so it cannot be combined with secure aggregation.\n");
    return 0;
}


Appending to fedml.cu


## Step 3 - Compile
`nvcc` compiles the CUDA and C++ code together into one executable.

In [11]:
!nvcc -O2 -std=c++17 fedml.cu -o fedml

## Step 4 - Run all experiments
Prints four result blocks: **1a** correctness, **1b** speed, **2** DP + secure aggregation, **3** model poisoning.

In [12]:
!./fedml /kaggle/working/data

GPU: Tesla T4
Data: 3882 train, 624 test

=== 1a. Correctness ===
DP gradient (clip=1): max relative error 2.1e-07  OK
DP gradient (clip=0): max relative error 7.3e-08  OK
Secure aggregation: max error vs plain mean 7.4e-06 (fixed-point bound 7.6e-06)
Secure aggregation: serial and CUDA bit-identical: yes

=== 1b. Speed (best of 3, ms) ===
task                            serial      CUDA   speedup
DP gradient, N=1000               1.50      0.52      2.9x
DP gradient, N=10000             15.68      6.82      2.3x
DP gradient, N=100000           157.11     67.86      2.3x
Secure agg, 100000 params         6.58      0.61     10.8x
Secure agg, 1000000 params       69.83      5.26     13.3x
Secure agg, 5000000 params      360.37     26.34     13.7x

=== 2. Federated training with DP + secure aggregation ===
setting                   epsilon  acc serial  acc CUDA time serial time CUDA  speedup
no privacy                    inf      0.7228    0.7228       0.78s     0.09s     8.8x
DP sigma=1 

## How to read the results
- **1a Correctness:** the CUDA gradient should match the serial one (tiny relative error), and secure aggregation should reproduce the plain mean up to fixed-point rounding, bit-identical on CPU and GPU.
- **1b Speed:** CUDA wins more as the batch size or model size grows; for small inputs the copy overhead dominates.
- **2 Privacy:** larger sigma means more noise, so a smaller epsilon (stronger privacy) but lower accuracy. Serial and CUDA accuracies should be nearly equal.
- **3 Poisoning:** a mean (and therefore secure aggregation) is dragged by one malicious client, while the median ignores it. The catch is that the median needs to see individual updates, so you cannot have secure aggregation and median together.